In [19]:
import os
import re
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [20]:
DATA_FOLDER = "data"
MODEL_FOLDER = "models"
OUTPUT_FOLDER = "outputs"

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

TRAINING_OUTPUT_FILE = "outputs/financial_headline_training_data.xlsx"
MODEL_OUTPUT_FILE = "models/financial_headline_tfidf_classifier.pkl"

In [21]:
def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s&/-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [22]:
CATEGORY_KEYWORDS = {
    "P&L": [
        "p&l",
        "profit and loss",
        "profit & loss",
        "statement of profit"
    ],

    "BS": [
        "balance sheet",
        "balance statement",
        "assets and liabilities",
        "bs"
    ],

    "CFS": [
        "cash flow",
        "cash flow statement",
        "statement of cash flows",
        "cfs"
    ],

    "SR": [
        "segment revenue",
        "segment reporting",
        "segment information",
        "sr"
    ]
}

In [23]:
TYPE_KEYWORDS = {
    "Standalone": [
        "standalone"
    ],

    "Consolidated": [
        "consolidated"
    ]
}

In [24]:
def get_labels_from_sheet_name(sheet_name):

    sheet = str(sheet_name).strip().lower()

    category = None
    report_type = None

    for cat, keywords in CATEGORY_KEYWORDS.items():
        if any(keyword in sheet for keyword in keywords):
            category = cat
            break

    for typ, keywords in TYPE_KEYWORDS.items():
        if any(keyword in sheet for keyword in keywords):
            report_type = typ
            break

    return category, report_type

In [25]:
def find_particulars_column(df):
    for col in df.columns:
        if str(col).strip().lower() == "particulars":
            return col
    return None

In [26]:
def create_training_dataset_page_level(folder_path):
    all_rows = []

    for root, dirs, files in os.walk(folder_path):

        for file_name in files:

            if not file_name.lower().endswith((".xlsx", ".xls")):
                continue

            file_path = os.path.join(root, file_name)
            relative_path = os.path.relpath(file_path, folder_path)

            print(f"Reading file: {relative_path}")

            try:
                excel_file = pd.ExcelFile(file_path)

                for sheet_name in excel_file.sheet_names:

                    category, report_type = get_labels_from_sheet_name(sheet_name)

                    if category is None or report_type is None:
                        print(f"Skipping sheet: {sheet_name}")
                        continue

                    df = pd.read_excel(
                        file_path,
                        sheet_name=sheet_name,
                        engine="openpyxl"
                    )

                    particulars_col = find_particulars_column(df)

                    if particulars_col is None:
                        print(
                            f"Particulars column not found in "
                            f"{relative_path} - {sheet_name}"
                        )
                        continue

                    particulars_values = (
                        df[particulars_col]
                        .dropna()
                        .astype(str)
                        .tolist()
                    )


                    combined_text = (
                        sheet_name + " " +
                        " ".join(particulars_values)
                    )

                    cleaned_combined_text = clean_text(combined_text)

                    if cleaned_combined_text == "":
                        continue

                    combined_label = f"{category}_{report_type}"

                    all_rows.append({
                        "Combined_Text": combined_text,
                        "Cleaned_Combined_Text": cleaned_combined_text,
                        "Category": category,
                        "Report_Type": report_type,
                        "Combined_Label": combined_label,
                        "Source_File": file_name,
                        "Source_Path": relative_path,
                        "Source_Sheet": sheet_name
                    })

            except Exception as e:
                print(f"Error reading {relative_path}: {e}")

    return pd.DataFrame(all_rows)


In [27]:
training_df = create_training_dataset_page_level(DATA_FOLDER)

training_df.head()

Reading file: Banking Excel Files_20260603\AU Small Finance Bank Ltd_REMAPPED.xlsx
Skipping sheet: Segment Finance - Standalone
Reading file: Banking Excel Files_20260603\AXIS Bank Ltd_REMAPPED.xlsx
Skipping sheet: Segment Result_Standalone
Skipping sheet: Segment Result_Consolidated
Reading file: Banking Excel Files_20260603\Bandhan Bank Ltd_REMAPPED.xlsx
Skipping sheet: Segment Finance - Standalone
Reading file: Banking Excel Files_20260603\Bank Of Baroda_REMAPPED.xlsx
Skipping sheet: Segment Finance - Standalone
Skipping sheet: Segment Finance - Consolidated
Reading file: Banking Excel Files_20260603\Bank of India_REMAPPED.xlsx
Reading file: Banking Excel Files_20260603\Bank of maharashtra_REMAPPED.xlsx
Skipping sheet: Segment Finance - Standalone
Skipping sheet: Segment Finance - Consolidated
Reading file: Banking Excel Files_20260603\Canara Bank_REMAPPED.xlsx
Skipping sheet: Segment Finance - Standalone
Skipping sheet: Segment Finance - Consolidated
Reading file: Banking Excel Fil

,Combined_Text,Cleaned_Combined_Text,Category,Report_Type,Combined_Label,Source_File,Source_Path,Source_Sheet
0,Standalone Profit & Loss Interest Earned (a)+(...,standalone profit & loss interest earned a b c...,P&L,Standalone,P&L_Standalone,AU Small Finance Bank Ltd_REMAPPED.xlsx,Banking Excel Files_20260603\AU Small Finance ...,Standalone Profit & Loss
1,Standalone Balance Sheet CAPITAL & LIABILITIES...,standalone balance sheet capital & liabilities...,BS,Standalone,BS_Standalone,AU Small Finance Bank Ltd_REMAPPED.xlsx,Banking Excel Files_20260603\AU Small Finance ...,Standalone Balance Sheet
2,Standalone Cash Flow Profit after tax Add: Pro...,standalone cash flow profit after tax add prov...,CFS,Standalone,CFS_Standalone,AU Small Finance Bank Ltd_REMAPPED.xlsx,Banking Excel Files_20260603\AU Small Finance ...,Standalone Cash Flow
3,Standalone Profit & Loss Interest earned Inter...,standalone profit & loss interest earned inter...,P&L,Standalone,P&L_Standalone,AXIS Bank Ltd_REMAPPED.xlsx,Banking Excel Files_20260603\AXIS Bank Ltd_REM...,Standalone Profit & Loss
4,Consolidated Profit & Loss Interest earned Int...,consolidated profit & loss interest earned int...,P&L,Consolidated,P&L_Consolidated,AXIS Bank Ltd_REMAPPED.xlsx,Banking Excel Files_20260603\AXIS Bank Ltd_REM...,Consolidated Profit & Loss


In [28]:
training_df["Category"].value_counts()



Category
P&L    842
BS     835
CFS    835
SR     484
Name: count, dtype: int64

In [29]:
training_df.to_excel(
    TRAINING_OUTPUT_FILE,
    index=False,
    engine="openpyxl"
)

print(f"Training dataset saved to: {TRAINING_OUTPUT_FILE}")

Training dataset saved to: outputs/financial_headline_training_data.xlsx


In [30]:
df = training_df.copy()

combined_df = training_df.dropna(
    subset=["Cleaned_Combined_Text", "Combined_Label"]
)

X = combined_df["Cleaned_Combined_Text"]
y = combined_df["Combined_Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

combined_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 3),
        max_features=20000,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

combined_model.fit(X_train, y_train)

print("Combined 8-class model trained successfully.")

Combined 8-class model trained successfully.


In [31]:
y_pred = combined_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.8883333333333333

                  precision    recall  f1-score   support

 BS_Consolidated       1.00      0.96      0.98        78
   BS_Standalone       0.97      1.00      0.98        89
CFS_Consolidated       0.85      0.67      0.75        78
  CFS_Standalone       0.75      0.90      0.82        89
P&L_Consolidated       0.97      0.72      0.83        79
  P&L_Standalone       0.80      0.98      0.88        90
 SR_Consolidated       0.92      0.98      0.95        50
   SR_Standalone       0.98      0.91      0.95        47

        accuracy                           0.89       600
       macro avg       0.91      0.89      0.89       600
    weighted avg       0.90      0.89      0.89       600



In [32]:
pd.crosstab(
    y_test,
    y_pred,
    rownames=["Actual"],
    colnames=["Predicted"]
)


Predicted,BS_Consolidated,BS_Standalone,CFS_Consolidated,CFS_Standalone,P&L_Consolidated,P&L_Standalone,SR_Consolidated,SR_Standalone
Actual,,,,,,,,
BS_Consolidated,75,3,0,0,0,0,0,0
BS_Standalone,0,89,0,0,0,0,0,0
CFS_Consolidated,0,0,52,26,0,0,0,0
CFS_Standalone,0,0,9,80,0,0,0,0
P&L_Consolidated,0,0,0,0,57,22,0,0
P&L_Standalone,0,0,0,0,2,88,0,0
SR_Consolidated,0,0,0,0,0,0,49,1
SR_Standalone,0,0,0,0,0,0,4,43


In [33]:
joblib.dump(combined_model, "models/financial_combined_8class_classifier.pkl")

print(f"Model saved to: {MODEL_OUTPUT_FILE}")

Model saved to: models/financial_headline_tfidf_classifier.pkl
